In [1]:
%cd
%cd dev1/scaling-laws-ecnn/

/home/frischs
/home/frischs/dev1/scaling-laws-ecnn


In [18]:
import numpy as np
import os
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import torch
from PIL import Image
from torchvision import transforms

In [3]:
location = "../../Data/frischs/datasets/cifar10/CIFAR-10-C/"

In [15]:
%ls ../../Data/frischs/datasets/cifar10/CIFAR-10-C/

brightness.npy         gaussian_noise.npy    saturate.npy
contrast.npy           glass_blur.npy        shot_noise.npy
defocus_blur.npy       impulse_noise.npy     snow.npy
elastic_transform.npy  jpeg_compression.npy  spatter.npy
fog.npy                labels.npy            speckle_noise.npy
frost.npy              motion_blur.npy       zoom_blur.npy
gaussian_blur.npy      pixelate.npy


In [21]:
np.load(location + 'labels.npy').shape

(50000,)

In [27]:
files = os.listdir(location)
files = [f.split(".")[0] for f in files if "labels" not in f]
files

['speckle_noise',
 'defocus_blur',
 'brightness',
 'frost',
 'jpeg_compression',
 'glass_blur',
 'gaussian_blur',
 'elastic_transform',
 'shot_noise',
 'spatter',
 'fog',
 'contrast',
 'zoom_blur',
 'saturate',
 'motion_blur',
 'gaussian_noise',
 'pixelate',
 'impulse_noise',
 'snow']

In [37]:
class CIFAR10_C(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.files = os.listdir(location)
        self.files = [f.split(".")[0] for f in self.files if "labels" not in f]
        
        self.test_images_under_dif_perturbations = []
        self.test_labels_under_dif_perturbations = []
        self.meta_data_list = []
        for i, file in enumerate(self.files):
            self.test_images_under_dif_perturbations.append(np.load(location + file + '.npy'))
            self.test_labels_under_dif_perturbations.append(np.load(location + 'labels.npy'))
            self.meta_data_list.append([i] * len(self.test_labels_under_dif_perturbations[-1]))
        
        self.test_images_under_dif_perturbations = np.concatenate(self.test_images_under_dif_perturbations)
        self.test_labels_under_dif_perturbations = np.concatenate(self.test_labels_under_dif_perturbations)
        self.meta_data_list = np.concatenate(self.meta_data_list)

        print("test_images_under_dif_perturbations.shape: ", self.test_images_under_dif_perturbations.shape)
        print("test_labels_under_dif_perturbations.shape: ", self.test_labels_under_dif_perturbations.shape)
        print("meta_data_list.shape: ", self.meta_data_list.shape)
        print("meta_data_list: ", self.meta_data_list[:10])

        
    def __len__(self):
        return len(self.test_images_under_dif_perturbations)
    
    def __getitem__(self, index):
        image = self.test_images_under_dif_perturbations[index]
        label = self.test_labels_under_dif_perturbations[index]
        meta_data = self.meta_data_list[index]
        
        image = Image.fromarray(image)
        
        if self.transform is not None:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.int64)
        meta_data = torch.tensor(meta_data, dtype=torch.int64)
        
        return image, label, meta_data
    
    def eval(self, predictions, labels, meta_data_list):
        # predictions: (N,)
        # labels: (N,)
        # meta_data_list: (N,)
        # returns: (num_perturbations, num_classes)
        assert len(predictions) == len(labels) == len(meta_data_list)
        
        perturbations = np.unique(meta_data_list)
        
        # compute accuracy for each perturbation
        accs = {}
        for perturbation in perturbations:
            idx = meta_data_list == perturbation
            print("idx.shape: ", idx.shape)
            accs[f"acc_{self.files[perturbation]}"] = (predictions[idx] == labels[idx]).mean()

        # compute average accuracy over all perturbations
        accs["mCE"] = np.mean(list(accs.values()))

        return accs


dataset_cifar10_c= CIFAR10_C(location, transforms.ToTensor())
"""
test_loader = DataLoader(dataset_cifar10_c, batch_size=4048, shuffle=False, num_workers=8)


predictions_all = []
labels_all = []
meta_data_all = []
for i, (images, labels, meta_data_list) in enumerate(test_loader):
    prediction = np.random.randint(0, 10, (len(images),))
    predictions_all.append(prediction)
    labels_all.append(labels.numpy())
    meta_data_all.append(meta_data_list.numpy())

predictions_all = np.concatenate(predictions_all)
labels_all = np.concatenate(labels_all)
meta_data_all = np.concatenate(meta_data_all)

dataset_cifar10_c.eval(predictions_all, labels_all, meta_data_all)
"""


test_images_under_dif_perturbations.shape:  (950000, 32, 32, 3)
test_labels_under_dif_perturbations.shape:  (950000,)
meta_data_list.shape:  (950000,)
meta_data_list:  [0 0 0 0 0 0 0 0 0 0]


'\ntest_loader = DataLoader(dataset_cifar10_c, batch_size=4048, shuffle=False, num_workers=8)\n\n\npredictions_all = []\nlabels_all = []\nmeta_data_all = []\nfor i, (images, labels, meta_data_list) in enumerate(test_loader):\n    prediction = np.random.randint(0, 10, (len(images),))\n    predictions_all.append(prediction)\n    labels_all.append(labels.numpy())\n    meta_data_all.append(meta_data_list.numpy())\n\npredictions_all = np.concatenate(predictions_all)\nlabels_all = np.concatenate(labels_all)\nmeta_data_all = np.concatenate(meta_data_all)\n\ndataset_cifar10_c.eval(predictions_all, labels_all, meta_data_all)\n'

In [38]:
dataset_cifar10_c.eval(predictions_all, labels_all, meta_data_all)

idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)
idx.shape:  (950000,)


{'speckle_noise': 0.10022,
 'defocus_blur': 0.10098,
 'brightness': 0.10222,
 'frost': 0.0995,
 'jpeg_compression': 0.10198,
 'glass_blur': 0.10164,
 'gaussian_blur': 0.10152,
 'elastic_transform': 0.10176,
 'shot_noise': 0.10018,
 'spatter': 0.1005,
 'fog': 0.09976,
 'contrast': 0.09912,
 'zoom_blur': 0.0965,
 'saturate': 0.1005,
 'motion_blur': 0.10092,
 'gaussian_noise': 0.10082,
 'pixelate': 0.0978,
 'impulse_noise': 0.09838,
 'snow': 0.10014,
 'mCE': 0.1002336842105263}